#### 1. 원유_환율 데이터 정보

In [43]:
import FinanceDataReader as fdr
import pandas as pd

def fetch_oil_fx_fdr(start="2010-01-01"):
    """
    FDR 기반 유가 + 환율 수집 (방어 로직 포함)
    """
    data = []

    series_map = {
        "oil_wti": "CL=F",     # WTI
        "oil_brent": "BZ=F",  # Brent
        "usdkrw": "USD/KRW"
    }

    for indicator, symbol in series_map.items():
        df = fdr.DataReader(symbol, start)

        # 1️⃣ index → date
        df = df.reset_index()

        # 2️⃣ date 컬럼명 통일
        if 'Date' in df.columns:
            df.rename(columns={'Date': 'date'}, inplace=True)
        elif 'date' not in df.columns:
            df.rename(columns={df.columns[0]: 'date'}, inplace=True)

        # 3️⃣ 가격 컬럼 선택 (Close → Price)
        if 'Close' in df.columns:
            value_col = 'Close'
        elif 'Price' in df.columns:
            value_col = 'Price'
        else:
            raise ValueError(f"[ERROR] 가격 컬럼 없음: {symbol}")

        df = df[['date', value_col]].rename(columns={value_col: 'value'})
        df['indicator'] = indicator
        df['source'] = 'FDR'

        data.append(df)

    return pd.concat(data, ignore_index=True)


# 실행
df_macro = fetch_oil_fx_fdr("2010-01-01")
df_macro.tail()


,date,value,indicator,source
12306,2026-01-30,1428.760010,usdkrw,FDR
12307,2026-02-02,1448.800049,usdkrw,FDR
12308,2026-02-03,1452.540039,usdkrw,FDR
12309,2026-02-04,1446.680054,usdkrw,FDR
12310,2026-02-05,1465.140015,usdkrw,FDR


In [44]:
# df_macro.to_csv(r'C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\KS_Air_Tourism\mactor_sample.csv')

#### 2. 여행지출전망 (BOK)

In [13]:
import requests

BASE = "https://ecos.bok.or.kr/api"

def ecos_stat_search(api_key: str,
                     stat_code: str,
                     cycle: str,
                     start_date: str,
                     end_date: str,
                     item_code1: str = "",
                     item_code2: str = "",
                     item_code3: str = "",
                     lang: str = "kr",
                     timeout: int = 30) -> pd.DataFrame:
    """
    ECOS StatisticSearch 호출 (JSON)
    - cycle: D/M/Q/A
    - start_date/end_date: M이면 YYYYMM (예: 201001)
    """
    url = "/".join([
        BASE, "StatisticSearch",
        api_key, "json", lang,
        "1", "100000",
        stat_code, cycle, start_date, end_date,
        item_code1, item_code2, item_code3
    ])

    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    js = r.json()

    rows = js.get("StatisticSearch", {}).get("row", [])
    df = pd.DataFrame(rows)
    if df.empty:
        return df

    # 표준화
    df = df.rename(columns={
        "TIME": "date",
        "DATA_VALUE": "value",
        "STAT_CODE": "stat_code",
        "STAT_NAME": "stat_name",
        "ITEM_CODE1": "item_code1",
        "ITEM_NAME1": "item_name1",
        "ITEM_CODE2": "item_code2",
        "ITEM_NAME2": "item_name2",
        "ITEM_CODE3": "item_code3",
        "ITEM_NAME3": "item_name3",
        "UNIT_NAME": "unit"
    })

    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    # M 주기면 'YYYYMM' → Timestamp(월말)로 변환 (원하시면 월초로도 바꿀 수 있음)
    if cycle == "M":
        df["date"] = pd.to_datetime(df["date"], format="%Y%m") + pd.offsets.MonthEnd(0)
    else:
        # 다른 주기는 일단 원문 유지 (필요 시 추가 변환)
        df["date"] = df["date"].astype(str)

    return df


def fetch_travel_spending_expectation_csi_total(api_key: str,
                                               start_date: str = "200809",
                                               end_date: str = "202512") -> pd.DataFrame:
    """
    소비자동향조사 - 여행비 지출전망 CSI (전체)
    """
    stat_code = "511Y002"
    cycle = "M"
    item_code1 = "FMCCD"   # 여행비 지출전망CSI
    item_code2 = "99988"   # 전체

    df = ecos_stat_search(
        api_key=api_key,
        stat_code=stat_code,
        cycle=cycle,
        start_date=start_date,
        end_date=end_date,
        item_code1=item_code1,
        item_code2=item_code2
    )

    if df.empty:
        raise RuntimeError("ECOS 응답이 비어 있습니다. (코드/기간/키 확인 필요)")

    # 지표상회 long-format으로 변환
    out = df[["date", "value"]].copy()
    out["indicator"] = "csi_travel_spending_expectation_total"  # 프로젝트 표준명(원하면 한글로)
    out["source"] = "BOK_ECOS"
    return out.sort_values("date").reset_index(drop=True)


# =========================
# 실행
# =========================

from DATA.KEYS import *

api_key = KEYS['BOK']

BOK_API_KEY = api_key
df_travel_csi = fetch_travel_spending_expectation_csi_total(
    api_key=BOK_API_KEY,
    start_date="200809",
    end_date="202512"
)

print(df_travel_csi.tail())


          date  value                              indicator    source
203 2025-08-31     98  csi_travel_spending_expectation_total  BOK_ECOS
204 2025-09-30     97  csi_travel_spending_expectation_total  BOK_ECOS
205 2025-10-31     97  csi_travel_spending_expectation_total  BOK_ECOS
206 2025-11-30     98  csi_travel_spending_expectation_total  BOK_ECOS
207 2025-12-31     97  csi_travel_spending_expectation_total  BOK_ECOS


In [45]:
# df_travel_csi.to_csv(r'C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\KS_Air_Tourism\tour_csi_sample.csv')

In [22]:
from typing import Optional, Dict
from datetime import datetime

BASE = "https://apis.data.go.kr/B551177/AviationStatsByAirline"

AIRLINES = {
    "KE": "대한항공",
    "OZ": "아시아나항공",
    "7C": "제주항공",
    "AA": "아메리칸항공",
}

def _normalize_json(js):
    """JSON 최상위가 list면 첫 원소(dict)로 정규화"""
    if isinstance(js, list):
        return js[0] if js else {}
    return js if isinstance(js, dict) else {}

def _extract_items(js):
    """
    response > body > items > item 을 최대한 안전하게 추출
    - items가 dict일 수도, list일 수도 있음
    - item이 dict일 수도, list일 수도 있음
    """
    js = _normalize_json(js)

    # response가 dict일 수도, list일 수도 있음
    resp = js.get("response", js)
    resp = _normalize_json(resp)

    body = resp.get("body", resp)
    body = _normalize_json(body)

    # body가 dict가 아니면 포기
    if not isinstance(body, dict):
        return []

    # items 블록은 dict 또는 list로 올 수 있음
    items_block = body.get("items", None)

    # 케이스 A: {"items": {"item": [...]}}
    if isinstance(items_block, dict):
        items = items_block.get("item", [])
    # 케이스 B: {"items": [...]}
    elif isinstance(items_block, list):
        items = items_block
    else:
        # 혹시 body에 item이 바로 있는 케이스 방어
        items = body.get("item", [])

    # item이 dict면 리스트로
    if items is None:
        items = []
    if isinstance(items, dict):
        items = [items]

    # 혹시 [{"item":[...]}] 처럼 한 번 더 감싼 경우 방어
    if isinstance(items, list) and len(items) == 1 and isinstance(items[0], dict) and "item" in items[0]:
        nested = items[0]["item"]
        if isinstance(nested, list):
            items = nested
        elif isinstance(nested, dict):
            items = [nested]

    return items



def month_range(start_yyyymm: str, end_yyyymm: str):
    """YYYYMM 문자열 범위 생성 (포함)"""
    sy, sm = int(start_yyyymm[:4]), int(start_yyyymm[4:])
    ey, em = int(end_yyyymm[:4]), int(end_yyyymm[4:])
    y, m = sy, sm
    while (y < ey) or (y == ey and m <= em):
        yield f"{y:04d}{m:02d}"
        m += 1
        if m == 13:
            y += 1
            m = 1

def _call_api(endpoint: str, params: dict, timeout: int = 30) -> dict:
    """
    공공데이터포털 API 호출 (JSON)
    - 오류 응답이 XML(OpenAPI_ServiceResponse)로 올 수도 있어 예외 메시지를 명확히 함
    """
    url = f"{BASE}/{endpoint}"
    r = requests.get(url, params=params, timeout=timeout)

    # HTTP 자체 오류
    try:
        r.raise_for_status()
    except Exception as e:
        raise RuntimeError(f"HTTP error: {e} / url={r.url} / body={r.text[:300]}")

    # JSON 파싱 시도
    try:
        js = r.json()
        return js
    except Exception:
        # JSON이 아니면(대개 오류 XML) 텍스트 일부 보여줌
        raise RuntimeError(f"Non-JSON response (maybe XML error). url={r.url} / body={r.text[:400]}")

def fetch_monthly_passenger(service_key: str, yyyymm: str) -> pd.DataFrame:
    params = {
        "serviceKey": service_key,
        "from_month": yyyymm,
        "to_month": yyyymm,
        "type": "json",
    }

    js = _call_api("getTotalNumberOfPassenger", params=params)
    items = _extract_items(js)

    df = pd.DataFrame(items)
    if df.empty:
        return df

    for c in ["arrPassenger", "depPassenger", "passenger"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", "", regex=False), errors="coerce")

    df["yyyymm"] = yyyymm
    df["metric"] = "passenger"
    return df

def fetch_monthly_cargo(service_key: str, yyyymm: str) -> pd.DataFrame:
    params = {
        "serviceKey": service_key,
        "from_month": yyyymm,
        "to_month": yyyymm,
        "type": "json",
    }

    js = _call_api("getTotalTonsOfCargo", params=params)
    items = _extract_items(js)

    df = pd.DataFrame(items)
    if df.empty:
        return df

    for c in ["arrBaggage", "depBaggage", "baggage"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", "", regex=False), errors="coerce")

    df["yyyymm"] = yyyymm
    df["metric"] = "cargo_tons"
    return df

def fetch_airline_monthly_stats(
    service_key: str,
    start_yyyymm: str = "202201",
    end_yyyymm: Optional[str] = None,
    airlines: Optional[Dict[str, str]] = None
) -> pd.DataFrame:
    """
    2022년 이후 월별 여객/화물 데이터를 모아 long-format으로 반환
    """
    if airlines is None:
        airlines = AIRLINES

    if end_yyyymm is None:
        end_yyyymm = datetime.today().strftime("%Y%m")

    frames = []
    for yyyymm in month_range(start_yyyymm, end_yyyymm):
        df_p = fetch_monthly_passenger(service_key, yyyymm)
        df_c = fetch_monthly_cargo(service_key, yyyymm)

        for df in [df_p, df_c]:
            if df is None or df.empty:
                continue

            if "airlineCode" in df.columns:
                df = df[df["airlineCode"].isin(airlines.keys())].copy()
            else:
                continue

            frames.append(df)

    if not frames:
        return pd.DataFrame()

    out = pd.concat(frames, ignore_index=True)
    out["date"] = pd.to_datetime(out["yyyymm"], format="%Y%m") + pd.offsets.MonthEnd(0)
    out["airlineName_kr"] = out["airlineCode"].map(airlines)

    return out.sort_values(["date", "airlineCode"]).reset_index(drop=True)



# =========================
# 사용 예시
# # =========================
SERVICE_KEY = KEYS['ODPD']  # 공공데이터포털 'Decoding' 키 권장
#
df = fetch_airline_monthly_stats(
    service_key=SERVICE_KEY,
    start_yyyymm="202201",
    end_yyyymm=None,   # None이면 이번 달까지
)

print(df.tail(20))

# 저장
# df.to_csv("airline_monthly_pax_cargo_2022plus.csv", index=False, encoding="utf-8-sig")


    airline airlineCode  arrPassenger  depPassenger  passenger  yyyymm  \
364    대한항공          KE      770746.0      739777.0  1510523.0  202510   
365    대한항공          KE           NaN           NaN        NaN  202510   
366  아시아나항공          OZ      475743.0      453383.0   929126.0  202510   
367  아시아나항공          OZ           NaN           NaN        NaN  202510   
368    제주항공          7C      246555.0      252076.0   498631.0  202511   
369    제주항공          7C           NaN           NaN        NaN  202511   
370  아메리칸항공          AA        7069.0        7579.0    14648.0  202511   
371  아메리칸항공          AA           NaN           NaN        NaN  202511   
372    대한항공          KE      714564.0      734047.0  1448611.0  202511   
373    대한항공          KE           NaN           NaN        NaN  202511   
374  아시아나항공          OZ      418328.0      430910.0   849238.0  202511   
375  아시아나항공          OZ           NaN           NaN        NaN  202511   
376    제주항공          7C      269781.0 

In [47]:
# df.to_csv(r'C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\KS_Air_Tourism\ICN_stats.csv')